# AI in Plant Health — Colab Exam
## 6 Questions | 35 points total

**Instructions — read before you start**
- Answer **all six** questions. Points per question are shown in each header.
- Cells marked **RUN THIS CELL** must be run in order, top to bottom, before you answer.
- Some questions include an **interactive tool** (dropdowns / sliders). Use it to explore, then write your conclusions in the **YOUR ANSWER** box for that question.
- Grading rewards **biological reasoning**, not just the numbers the code prints. Explain *why*, not only *what*.
- Keep answers concise (2–5 sentences per part unless a part asks for a list).

**RUN THIS CELL**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, accuracy_score
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
import ipywidgets as widgets
from IPython.display import display, clear_output
np.random.seed(42)
print("Setup complete.")

Setup complete.


## Question 1 — Understanding the setup cell (3 points)

Look at the **RUN THIS CELL** code cell above (the one that imports libraries and prints `Setup complete.`).

Answer all three parts:
1. **What is the purpose of this cell**, and **why must it be run before any other cell** in the notebook? (1 pt)
2. **Pick at least five lines** from that cell and briefly explain what each one does (e.g., what a specific `import` brings in, what `np.random.seed(42)` controls). (1 pt)
3. In one sentence, what would happen if you **skipped this cell** and ran Question 4 directly? (1 pt)

>**Hint:** Think about *dependencies* — later cells call names like `np`, `pd`, `train_test_split`, and `widgets`. Where do those names come from? For part 2, `as np` / `as pd` create short aliases, and a fixed random seed makes results reproducible.


---
###Your Answer — Question 1
*Double-click this cell to edit, then type your answer after each label.*

**1.**  

**2.**  

**3.**  


---
## Question 2 — From a biological question to ML vocabulary (6 points)

A researcher wants to predict whether a rice plant is **disease-resistant or susceptible** using four measurements per plant: **SNP genotype, average temperature, rainfall, and lesion length**.


Answer all three parts:
1. List the **features** (input variables). (1.5 pts)
2. Identify the **target / label** (what is being predicted). (1.5 pts)
3. Is this primarily a **classification, regression, or clustering** problem? **Explain your choice in one sentence.** (3 pts)

> **Hint:** Features are what you measure; the label is what you want to predict. For part 3, ask whether the label is a **category** (resistant/susceptible) or a **continuous number**, and whether the label is **given** during training.

---
### Your Answer — Question 2
*Double-click this cell to edit, then type your answer after each label.*

**1.**  

**2.**  

**3.**   


---
## Question 3 — Match each problem to the right learning approach (6 points)

**Task:** **Run the interactive cell below**. For each of the four plant-health problems, use the dropdown to select the most appropriate learning approach, then click **Check matches** to self-check.

Problems:
- Healthy vs diseased leaf
- Predict disease severity (%)
- Discover natural groups in gene-expression samples
- Learn irrigation actions from rewards

Approaches to choose from: **Classification · Regression · Clustering · Reinforcement learning**

- Getting all four matches correct is worth **4 pts** (the self-check reports your score out of 4).
- The written follow-up below is worth **2 pts**.


**RUN THIS CELL** (then use the dropdowns and click *Check matches*)


In [ ]:
# @title RUN THIS CELL — Question 3 matching tool
scenarios = {
    "Healthy vs diseased leaf": "Classification",
    "Predict disease severity (%)": "Regression",
    "Discover natural groups in gene-expression samples": "Clustering",
    "Learn irrigation actions from rewards": "Reinforcement learning",
}
opts = ["Classification","Regression","Clustering","Reinforcement learning"]
answers = {}
for i, s in enumerate(scenarios.keys()):
    display(widgets.HTML(value=f"<b>{i+1}. {s}</b>"))  # Display scenario as numbered, bold text
    dd = widgets.Dropdown(options=opts, description="Approach:", layout=widgets.Layout(width="95%"))
    answers[s] = dd
    display(dd)

btn = widgets.Button(description="Check matches")
out = widgets.Output()
def check_matches(_):
    with out:
        clear_output()
        score = 0
        for s, a in scenarios.items():
            ok = answers[s].value == a
            score += int(ok)
            print(("✓" if ok else "✗"), s, "→", answers[s].value)
        print(f"Self-check: {score}/4")
btn.on_click(check_matches)
display(btn, out)

HTML(value='<b>1. Healthy vs diseased leaf</b>')

Dropdown(description='Approach:', layout=Layout(width='95%'), options=('Classification', 'Regression', 'Cluste…

HTML(value='<b>2. Predict disease severity (%)</b>')

Dropdown(description='Approach:', layout=Layout(width='95%'), options=('Classification', 'Regression', 'Cluste…

HTML(value='<b>3. Discover natural groups in gene-expression samples</b>')

Dropdown(description='Approach:', layout=Layout(width='95%'), options=('Classification', 'Regression', 'Cluste…

HTML(value='<b>4. Learn irrigation actions from rewards</b>')

Dropdown(description='Approach:', layout=Layout(width='95%'), options=('Classification', 'Regression', 'Cluste…

Button(description='Check matches', style=ButtonStyle())

Output()

**Written follow-up (2 pts).** Choose **one** of the four problems above and describe:
- the **input data**,
- the **expected output**,
- whether **labels are required during training**,
- **one** suitable evaluation metric or evaluation strategy.

>**Hint:** Match the metric to the task — accuracy/precision/recall for classification, RMSE or R² for regression, silhouette for clustering. The metric that reports the **false-positive rate** comes from the **confusion matrix** . Only supervised tasks (classification, regression) need labels during training.


---
### Your Answer — Question 3
*Double-click this cell to edit, then type your answer after each label.*

**Answer:**  


---
## Question 4 — Recognizing data leakage in a train/test split (6 points)

An imaging study photographs each plant **8 times**, so there are 8 photos per plant. The goal is a model that tells **healthy from diseased plants**, and the first step is to split the photos into a **training set** and a **test set**.

The cells below build the dataset and then make **two different splits**:
- **Split A — random, photo-level:** every photo is sent to train or test at random.
- **Split B — grouped by plant:** each plant's 8 photos stay together (all in train *or* all in test).

**Run both cells and read the printout carefully.** **One of these splits has data leakage** — your job is to spot which one and explain it. (No model is trained here; this question is only about how the data is split.)


In [ ]:
# @title
# RUN THIS CELL — build the synthetic image dataset
# 60 plants, 8 photos each = 480 photos. Plants 0-29 are healthy, plants 30-59 are diseased.
rng = np.random.default_rng(7)
rows = []
n_plants = 60
for plant_id in range(n_plants):
    disease = int(plant_id >= 30)
    for img in range(8):
        # two example measurements per photo (the exact values are not important here —
        # in this question we focus on HOW the data is split, not on training a model)
        f1 = rng.normal(disease, 1.0)
        f2 = rng.normal(0.8 * disease, 1.0)
        rows.append([plant_id, img, f1, f2, disease])

img_df = pd.DataFrame(rows, columns=["plant_id", "image_id", "feature1", "feature2", "disease"])
display(img_df.head())
print("Total photos:", len(img_df), "| Unique plants:", img_df.plant_id.nunique(), "| Photos per plant: 8")

,plant_id,image_id,feature1,feature2,disease
0,0,0,0.001230,0.298746,0
1,0,1,-0.274138,-0.890592,0
2,0,2,-0.454671,-0.991647,0
3,0,3,0.060144,1.340215,0
4,0,4,-0.492207,-0.620475,0


Total photos: 480 | Unique plants: 60 | Photos per plant: 8


In [ ]:
# @title
# RUN THIS CELL — make the two splits and check them for leakage
# A plant "leaks" if some of its photos are in TRAIN and some are in TEST at the same time.
y = img_df["disease"]
groups = img_df["plant_id"]
all_photos = np.arange(len(img_df))

# Split A: random photo-level split (each photo is assigned independently)
trainA, testA = train_test_split(all_photos, test_size=0.25, random_state=42, stratify=y)

# Split B: plant-level group split (a whole plant goes to train OR test, never both)
trainB, testB = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42).split(all_photos, y, groups))

def plants_in_both(train_pos, test_pos):
    return sorted(set(groups.iloc[train_pos]) & set(groups.iloc[test_pos]))

sharedA = plants_in_both(trainA, testA)
sharedB = plants_in_both(trainB, testB)

print("SPLIT A — random, photo-level")
print(f"   plants with photos in BOTH train and test: {len(sharedA)} out of 60")
print(f"   example plant IDs found on both sides: {sharedA[:8]}")
print()
print("SPLIT B — grouped by plant")
print(f"   plants with photos in BOTH train and test: {len(sharedB)} out of 60")
print()

# Zoom in on ONE plant to see what leakage looks like:
pid = sharedA[0]
train_photos = sorted(img_df.iloc[trainA].query("plant_id == @pid")["image_id"])
test_photos  = sorted(img_df.iloc[testA].query("plant_id == @pid")["image_id"])
print(f"Example (Split A) — plant #{pid}: photos {train_photos} are in TRAIN, "
      f"while photos {test_photos} are in TEST — the SAME plant is on both sides!")

SPLIT A — random, photo-level
   plants with photos in BOTH train and test: 54 out of 60
   example plant IDs found on both sides: [0, 1, 2, 4, 6, 7, 8, 9]

SPLIT B — grouped by plant
   plants with photos in BOTH train and test: 0 out of 60

Example (Split A) — plant #0: photos [0, 2, 3, 4, 5, 6, 7] are in TRAIN, while photos [1] are in TEST — the SAME plant is on both sides!


Read the printout above, then answer all four parts:
1. **Which split (A or B) has data leakage?** Point to the exact number in the printout that tells you. (1 pt)
2. **What causes the leakage?** Explain why 8 photos of the *same* plant are **not independent** samples. (2 pts)
3. **Why is leakage a problem?** What happens to the test accuracy, and to the model's performance on **brand-new plants** it has never seen? (1.5 pts)
4. Name **one** other real-world grouping variable (besides plant) you would also keep together when splitting. (1.5 pts)

>**Hints:**
> - In Split A, 54 of 60 plants have photos in *both* train and test; in Split B that number is 0 — each plant sits on only one side.
> - Photos of one plant look almost identical, so testing the model on another photo of a plant it already trained on is like testing it on something it has already seen.
> - A model that has effectively "met" the test plants will look great on this test but can still fail on genuinely new plants in the field. Other things you might keep together: same field/plot, same date/batch, same greenhouse, same camera.


---
### Your Answer — Question 4
*Double-click this cell to edit, then type your answer after each label.*

**1.**  

**2.**  

**3.**  

**4.**  


---
## Question 5 — From a single neuron to a neural network (4 points)

A neural network is built from many small units called **neurons**. Understanding one neuron makes Question 6 easy to follow — there is **no code to run here**, just read and answer.

A single neuron takes its inputs, multiplies each by a **weight**, adds a **bias**, and passes the result through an **activation function**:

`output = f( w1·x1 + w2·x2 + b )`

Think of two plant measurements, **x1 = lesion size** and **x2 = leaf yellowing**:
- **Weights `w1`, `w2`** — how much each input matters (a **negative** weight pushes the output down).
- **Bias `b`** — the neuron's **threshold**: how easily it "fires".
- **Activation function `f`** — turns the weighted sum into the output (for example, **sigmoid** gives a value between 0 and 1, like a probability).

With two inputs, a single neuron can only split the data with a **flat, straight line**.

**Answer both questions below.**

**5.1 The parts of a neuron (2 pts).** A single neuron computes `output = f(w1·x1 + w2·x2 + b)`. In **one sentence each**, explain what the **weights (w1, w2)**, the **bias (b)**, and the **activation function (f)** each do.


**5.2 Why we need a network (2 pts).** A single neuron can only separate classes with a **straight line**. Explain why this is a problem for data whose two classes are **curved or interlocking**, and what a **neural network** does to solve it.


---
### Your Answer — Question 5
*Double-click this cell to edit, then type your answer after each label.*

**5.1**  

**5.2**  


---
## Question 6 — Neural-network architecture choices (10 points)

A neural network stacks many of the neurons from Question 5 into layers, which lets it draw **curved** decision boundaries. Run the **RUN THIS CELL** cell below to launch the interactive network trainer, then use the sliders/dropdowns to explore the four questions that follow. Points for each part are shown next to the question.

### Background — read before you start

**What the tool does.** It trains a small neural network (an **MLP**) on the `make_moons` toy dataset — two interlocking crescents that you **cannot** separate with a straight line, so the network must learn a **curved** boundary. The data is split into **training (70%)** and **validation (30%)**, and accuracy on both is measured after every epoch.

You can change **four** things and watch the effect:
- **Hidden layers / neurons** → the model's *capacity* (how complex a curve it can draw).
- **Learning rate** → how big each learning step is.
- **Epochs** → how many passes through the training data.
- **Activation** (`relu`, `tanh`, `sigmoid`) → the non-linearity inside each neuron.

**Reading the "Hidden Layers" menu.** The numbers list the neurons in each hidden layer, left to right — e.g. **(10, 5)** means first layer 10 neurons, second layer 5. More layers/neurons = more capacity (and more overfitting risk).

**Underfitting vs. overfitting — the key idea.**

| | Underfitting | Good fit | Overfitting |
|---|---|---|---|
| Cause | too simple / trained too little | just right | too complex / trained too long |
| Training accuracy | low | high | high |
| Validation accuracy | low | high | low (well below training) |
| Boundary shape | ~straight line | smooth curve | wiggly, chasing points |

The goal is the **middle column**: both accuracies high and close together.

**Reading the three plots.**
1. **Decision Boundary (left)** — fill color = true class; circle = training point, lime square = validation point; shaded area = the model's predicted region. A point is correct when its color matches the shaded region it sits in.
2. **Loss vs Epoch (middle)** — should drop then flatten. Still falling = needs more epochs; jumpy or rising = learning rate too high.
3. **Accuracy vs Epoch (right)** — two lines. Close and high = good fit; training climbs above validation (widening gap) = overfitting; both low = underfitting.

**Fastest check:** compare the two printed numbers. *Training ≈ Validation, both high* = good; *Training ≫ Validation* = overfitting; *both low* = underfitting.

Two classes :
Red and Blue

Divided in:
Traininig: Circle

Validation: Triangle **bold text**

In basic words: the model should draw a seperation that matches the color of the backgodun with points **bold text**

In [7]:
# @title
# RUN THIS CELL — interactive MLP trainer (TWO sets: training + validation)
from sklearn.datasets import make_moons
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.exceptions import ConvergenceWarning
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import warnings

# 1. Data -> only TWO sets: TRAINING (70%) and VALIDATION (30%)
X_synth, y_synth = make_moons(n_samples=200, noise=0.3, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_synth, y_synth, test_size=0.3, random_state=42)

# 2. Interactive widgets
hidden_layer_options = {
    "1 Hidden Layer (10 neurons)": (10,),
    "2 Hidden Layers (10, 5 neurons)": (10, 5),
    "3 Hidden Layers (10, 5, 2 neurons)": (10, 5, 2),
    "Large (200, 200 neurons)": (200, 200)   # try this + many epochs to SEE overfitting
}
hidden_layer_config_widget = widgets.Dropdown(
    options=hidden_layer_options, value=(10, 5), description='Hidden Layers:')

n_epochs_widget = widgets.IntSlider(
    min=50, max=500, step=50, value=200, description='Epochs:', continuous_update=False)

learning_rate_widget = widgets.FloatSlider(
    min=0.001, max=0.1, step=0.001, value=0.01,
    description='Learning Rate:', readout_format='.3f', continuous_update=False)

activation_widget = widgets.Dropdown(
    options=['relu', 'tanh', 'sigmoid'], value='relu', description='Activation:')

# 3. Train one epoch at a time, recording accuracy on BOTH sets after each epoch
def train_and_visualize_nn(hidden_layer_sizes, n_epochs, learning_rate_init, activation):
    clear_output(wait=True)

    # Map 'sigmoid' to 'logistic' (sklearn's name for the same function)
    actual_activation = 'logistic' if activation == 'sigmoid' else activation

    mlp = MLPClassifier(hidden_layer_sizes=hidden_layer_sizes,
                        learning_rate_init=learning_rate_init,
                        activation=actual_activation, solver='adam', random_state=42)

    classes = np.unique(y_train)
    train_acc, val_acc, loss_hist = [], [], []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        for _ in range(n_epochs):
            mlp.partial_fit(X_train, y_train, classes=classes)  # one epoch
            train_acc.append(mlp.score(X_train, y_train))
            val_acc.append(mlp.score(X_val, y_val))
            loss_hist.append(mlp.loss_)

    gap = train_acc[-1] - val_acc[-1]
    # Overfitting can mean a big gap OR near-perfect training accuracy with lower validation
    # (on this easy dataset validation caps ~0.9, so memorization shows as train ~1.0).
    if (train_acc[-1] >= 0.98 and gap >= 0.06) or gap > 0.10:
        verdict = "OVERFITTING (training well above validation -> memorizing)"
    elif val_acc[-1] < 0.70:
        verdict = "UNDERFITTING (both accuracies low)"
    else:
        verdict = "small gap, both high -> good fit"
    print(f"Settings: Hidden Layers={hidden_layer_sizes}, Epochs={n_epochs}, "
          f"LR={learning_rate_init}, Activation={activation}")
    print(f"\nFinal Training Accuracy:   {train_acc[-1]:.3f}")
    print(f"Final Validation Accuracy: {val_acc[-1]:.3f}")
    print(f"Overfitting gap (training - validation): {gap:+.3f}   ->   {verdict}")

    # Plotting
    fig, axes = plt.subplots(1, 3, figsize=(24, 6))

    # Plot 1: Decision Boundary
    h = .02
    x_min, x_max = X_synth[:, 0].min() - 0.5, X_synth[:, 0].max() + 0.5
    y_min, y_max = X_synth[:, 1].min() - 0.5, X_synth[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = mlp.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    axes[0].contourf(xx, yy, Z, cmap=plt.cm.RdBu, alpha=.4)
    axes[0].scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=plt.cm.RdBu, edgecolors='k', marker='o', s=50, label='Training (circle)')
    axes[0].scatter(X_val[:, 0], X_val[:, 1], c=y_val, cmap=plt.cm.RdBu, marker='^', s=130, label='Validation (triangle)')
    axes[0].set_title("Decision Boundary")
    axes[0].set_xlabel("Feature 1"); axes[0].set_ylabel("Feature 2"); axes[0].legend()

    # Plot 2: Training loss vs epoch
    axes[1].plot(range(1, n_epochs + 1), loss_hist, color='tab:purple')
    axes[1].set_title("Training Loss vs Epoch")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Training loss"); axes[1].grid(True)

    # Plot 3: Training vs Validation accuracy  --> the GAP between the lines = overfitting
    axes[2].plot(range(1, n_epochs + 1), train_acc, label='Training accuracy')
    axes[2].plot(range(1, n_epochs + 1), val_acc, label='Validation accuracy')
    axes[2].set_title("Accuracy vs Epoch  (gap between lines = overfitting)")
    axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Accuracy"); axes[2].set_ylim(0.4, 1.02)
    axes[2].legend(); axes[2].grid(True)

    fig.suptitle(
        "Left plot:  fill COLOR = true class   |   CIRCLE = training point, SQUARE (lime) = validation point   |   "
        "SHADED background = model's predicted region.        "
        "Right plot:  if the training line climbs well above the validation line, the model is OVERFITTING.",
        fontsize=12
    )
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.show()

# 4. Create the interactive interface
interactive_nn_trainer = widgets.interactive(
    train_and_visualize_nn,
    hidden_layer_sizes=hidden_layer_config_widget,
    n_epochs=n_epochs_widget,
    learning_rate_init=learning_rate_widget,
    activation=activation_widget
)
display(interactive_nn_trainer)

interactive(children=(Dropdown(description='Hidden Layers:', index=1, options={'1 Hidden Layer (10 neurons)': …

### Why two sets are enough to spot overfitting

- **Training set (70%)** — the data the model learns from.
- **Validation set (30%)** — held-out data the model never learns from, used only to check performance on unseen points.

After each epoch the tool prints the **gap = Training − Validation accuracy**. A large positive gap is the single number that reveals overfitting — you do **not** need a third (test) set to see it here.

**Answer in the boxes below, using the tool above.**

**6.1 More neurons → a curved boundary (3 pts).** Compare **"1 Hidden Layer (10 neurons)"** with **"Large (200, 200 neurons)"** at about 300 epochs. How does the **decision boundary** change, and can the bigger network separate the two crescents that a single neuron (Question 5) could not?

>**Hint:** One neuron draws a straight line

**6.2 Learning rate (2 pts).** Keep the architecture fixed (use Hidden Layer 10 neurons - don't change the layers) and compare a **very small** learning rate (0.001) with a **large** one (0.1), watching the **Training Loss vs Epoch** plot each time. What happens to the loss curve in each case, and why is a middle value usually best?


**6.3 Activation function (2 pts).** Keep the architecture fixed (use 2 Hidden Layer 10,5 neurons - don't change the layers) and switch the **Activation** between `relu`, `tanh`, and `sigmoid`. Which one gives the best result on this dataset, and what did you look at to decide?

>**Hint:** Compare the final **validation accuracy** and how quickly the loss curve drops for each. `sigmoid` often learns more slowly than `relu`/`tanh`.

**6.4 Overfitting (3 pts).** Pick **"Large (200, 200 neurons)"** with a **high number of epochs** and look at the **training vs validation accuracy** (right-hand plot).
- What happens to each of the two accuracies?
- How can you tell the network is **overfitting**, and name **one** thing you could do to reduce it?

>**Hint:** Overfitting = training accuracy climbs near 1.0 while validation stays lower (a big gap between the two lines). Ways to reduce it: use a smaller network, train for fewer epochs, or get more data.

---
### Your Answer — Question 6
*Double-click this cell to edit, then type your answer after each label.*

**6.1**  

**6.2**  

**6.3**  

**6.4**  